# transformers.js #1599 — where the Qwen3.5-4B prefill time goes

**What this notebook establishes:** the exported ONNX graph for `onnx-community/Qwen3.5-4B-ONNX`
evaluates its 24 gated-DeltaNet layers with a **sequential ONNX `Scan` over the sequence axis**
on the prefill path. Not a shader problem — a graph-structure problem.

**Why it's cheap to check:** the ONNX weights live in sibling `.onnx_data` files, so the *graph*
is only ~1.4 MB. No GPU, no auth, no 2.5 GB download. Runs in well under a minute on a free CPU runtime.

**What this notebook does _not_ claim.** It contains no timing. transformers.js runs WebGPU in a
browser; Colab is a Python VM with no browser and no WebGPU, so wall-clock numbers measured here
would not transfer. Everything below is a *static property of the published file* — deterministic,
and identical for anyone who runs it.

In [ ]:
%pip install -q onnx huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "onnx-community/Qwen3.5-4B-ONNX"
graph_path  = hf_hub_download(REPO, "onnx/decoder_model_merged_q4f16.onnx")  # ~1.4 MB, graph only
config_path = hf_hub_download(REPO, "config.json")
print(graph_path)

## 1. Op census — walking every subgraph

Ops hide inside `If`/`Scan` bodies, so a flat pass over `model.graph.node` undercounts. This recurses.

In [ ]:
import onnx, collections

model = onnx.load(graph_path, load_external_data=False)   # graph only; weights not needed

ops = collections.Counter()
def walk(g):
    for n in g.node:
        ops[n.op_type] += 1
        for a in n.attribute:
            if a.g and a.g.node:
                walk(a.g)
walk(model.graph)

print(f"Scan                          : {ops['Scan']:>3}   <- sequential recurrence")
print(f"GroupQueryAttention           : {ops['GroupQueryAttention']:>3}")
print(f"LinearAttention (ORT fused op) : {ops['LinearAttention']:>3}   <- absent from this graph")

## 2. Are those `Scan`s on the prefill path?

Each gated-DeltaNet layer emits an `If` switching on `/model/layers.N/gdn/is_decode`.
If every `Scan` sits in the **`else_branch`** (the not-decode side), the sequential recurrence
is specifically what a *prompt* pays.

In [ ]:
hits = []
def hunt(g):
    for n in g.node:
        if n.op_type == "If":
            cond = n.input[0] if n.input else "?"
            for a in n.attribute:
                if a.g and any(x.op_type == "Scan" for x in a.g.node):
                    hits.append((cond, a.name))
        for a in n.attribute:
            if a.g and a.g.node:
                hunt(a.g)
hunt(model.graph)

branches = collections.Counter(b for _, b in hits)
print(f"If-branches directly containing a Scan: {len(hits)}")
print(f"which branch: {dict(branches)}")
print("conditions (first 4):")
for c, b in hits[:4]:
    print(f"   {c}  ->  {b}")

## 3. What one `Scan` actually does

Its body is the gated delta rule; the scan axes say which dimension it walks.

In [ ]:
def first_scan(g):
    for n in g.node:
        if n.op_type == "Scan":
            return n
        for a in n.attribute:
            if a.g and a.g.node:
                r = first_scan(a.g)
                if r is not None:
                    return r
    return None

scan = first_scan(model.graph)
body = [a.g for a in scan.attribute if a.g][0]
axes = [list(a.ints) for a in scan.attribute if a.name == "scan_input_axes"]

print("body inputs      :", [i.name for i in body.input])
print("body node count  :", len(body.node))
print("body ops         :", collections.Counter(n.op_type for n in body.node).most_common())
print("scan_input_axes  :", axes, " <- axis 2 = sequence: one position per iteration")

## 4. Cross-check against `config.json`

The architecture should agree with the op census: 24 linear-attention layers, 8 full-attention.

In [ ]:
import json
cfg = json.load(open(config_path))
tc  = cfg.get("text_config", cfg)
lt  = collections.Counter(tc.get("layer_types", []))

print("layer_types            :", dict(lt))
print("full_attention_interval:", tc.get("full_attention_interval"))
print()
print(f"linear_attention layers == Scan count : {lt['linear_attention'] == ops['Scan']}")
print(f"full_attention layers   == GQA count  : {lt['full_attention']  == ops['GroupQueryAttention']}")

## 5. How the work scales with prompt length

Sequential body executions before the first token = `prompt_tokens x Scan_layers x body_nodes`.
These are *graph* quantities, not measurements — but they show the shape of the cost:
work grows linearly in prompt length and none of it is batched across positions.

In [ ]:
n_scan, n_body = ops["Scan"], len(body.node)
print(f"{'prompt tokens':>14} | {'sequential body-node executions':>32}")
print("-" * 50)
for t in (60, 252, 1024, 2044):
    print(f"{t:>14} | {t * n_scan * n_body:>32,}")
print(f"\n(= tokens x {n_scan} Scan layers x {n_body} nodes per body)")

## Summary

| finding | value |
|---|---|
| `Scan` ops (sequential recurrence) | **24** — one per `linear_attention` layer |
| `GroupQueryAttention` | 8 |
| `LinearAttention` (ORT fused contrib op) | **0** — this graph never uses it |
| Where the `Scan`s sit | `else_branch` of `If(/model/layers.N/gdn/is_decode)` → **prefill** |
| Scan body | 16 nodes, the gated delta rule |
| `scan_input_axes` | `[2,2,2,2,2]` → walks the sequence one position at a time |

Two consequences worth noting:

1. Since `LinearAttention` never appears, changes to that ORT kernel cannot affect this model —
   24 of its 32 layers are not attention at all. That is a plausible reason
   [microsoft/onnxruntime#27780](https://github.com/microsoft/onnxruntime/pull/27780) did not move the number.
2. The gated delta rule is *chunkable* — reference implementations evaluate a block of positions
   together and carry the state across chunks. If the export only emits the sequential form, this
   is a **prefill** issue arising in the ONNX export rather than in the runtime's shaders.

**Open question for maintainers:** is the sequential `Scan` intentional for this export, or is there
a chunked form that could be emitted instead?